# TEN anchor site: the structural test, and its answer

**Run all. CPU runtime. ~2 minutes.** No GPU needed.

## The answer (measured, not predicted)

Two independent entries place **Tyr176 within contact distance of the DNA** once the
substrate is long enough to reach the site:

| entry | Y176 → DNA | catalytic control | verdict |
|---|---|---|---|
| **7QXS** | **2.75 Å** (H-bond, OH···O3′) | 2.65 Å ✓ | **CONTACT** |
| **7QXA** | **4.35 Å** (vdW, OH···C5′) | 2.93 Å ✓ | **CONTACT** |
| 9SHY | 30.23 Å — but the relevant nucleotide is unmodelled | 2.78 Å ✓ | NOT RESOLVED |

Against **11.11–11.94 Å** in the 7-nucleotide reference structures. The contact belongs to
**Tyr176** and is to the **sugar–phosphate backbone**, not a base.

**This does not identify the mechanism.** A contact is equally consistent with a positioning
element and with a steric barrier. TPP1-stimulated RAP — mechanical or retention — stays open.

This notebook is a thin wrapper around files in the repository, which are the single source of
truth. It holds no logic of its own, so it cannot drift away from them.


In [ ]:
# --- 1. get the code -------------------------------------------------------
# If you uploaded this notebook on its own, clone the repo. If you opened it
# from the repo, this is a no-op.

import os, subprocess, sys

REPO_URL = 'https://github.com/Mr-Nobody-Anonymous/telomerase-allosteric-switch.git'
REPO_DIR = 'telomerase-allosteric-switch'

if not os.path.exists('colab/tpp1_colab.py'):
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print('working dir:', os.getcwd())
print('runner present:', os.path.exists('colab/tpp1_colab.py'))

In [ ]:
# --- 2. install dependencies ----------------------------------------------
!pip -q install numpy scipy gemmi

import numpy, scipy, gemmi
print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| gemmi', gemmi.__version__)

## 3. Self-test — run this first

Offline. Checks the script against the three bundled CIFs where the answer is already known
(PLYQ = 11.11 Å in 7TRE, 11.94 Å in 7TRD/7TRF), checks that TPP1 is found by **contact** and that
the histones in 7TRF are **not** mistaken for it, and exercises every verdict branch and the
RAP-state comparison with injected known motions.

**If this fails, stop. Do not run the download stage.** A failure here means the measurement
code is broken, and any result from stage 4 would be meaningless.

In [ ]:
!python colab/tpp1_colab.py --selftest

## 3b. Self-test — the verdict rule and this document

Two more offline suites. The first recomputes the table above from real coordinate subsets in
`colab/fixtures/`. The second checks that `AGENT_START_HERE.md` still matches the code.

In [ ]:
!python colab/verdict.py --selftest

In [ ]:
!python colab/test_colab.py

## 4. The real run — downloads 8 structures and measures

This fetches 7QXS, 7QXA, 9SHY, 9SHZ, 9SI0, plus the three local references, then measures:

* PLYQ / Y176 / TRAP-wedge (H752–H754) / anchor strand / IFD-TRAP / catalytic aspartates → DNA
* **every TEN-domain residue that contacts DNA**, per structure
* PLYQ distance **per nucleotide**, to see where along the DNA the anchor site sits
* TPP1 → DNA and TPP1 → PLYQ
* the three 2026 RAP states superposed on the RT ring, to see how far TEN and IFD-TRAP move
  **relative to the rigid ring** — not per-domain RMSD, which would absorb the motion

In [ ]:
!python colab/tpp1_colab.py

## 5. The report

Also written to `colab_out/report.md`. Paste this block back into the conversation — the verdict
line plus the distance table is enough.

In [ ]:
from pathlib import Path
p = Path('colab_out/report.md')
print(p.read_text() if p.exists() else 'No report produced — stage 4 must have failed.')

In [ ]:
# --- 6. per-nucleotide PLYQ profile (where along the DNA the anchor site sits) ---
import json
d = json.loads(Path('colab_out/results.json').read_text())
for s in d['structures']:
    prof = s.get('plyq_per_nt')
    if not prof:
        continue
    print(f"{s['pdb']} ({s['n_dna_nt']} nt, residues {s.get('dna_range')})")
    for row in prof:
        bar = '#' * max(0, int(30 - row['d'] * 2))
        print(f"   nt {row['nt']:>3d}  {row['d']:6.2f} A  {bar}")
    print()

## 7. What this cannot do

**It cannot settle the mechanism.** A backbone contact shows what is *reachable*, not which
route TPP1 uses, and it cannot distinguish retention from catalytic activation.

Specifically:

* **Do not quote 2.75 Å as a precise number.** These are 3.2–3.8 Å cryo-EM maps. What is robust
  is the ordering — ~3 Å against ~11 Å. The atom-pair assignment carries ~±1 Å.
* **Backbone, not base.** Y176 touches a phosphate/sugar oxygen, so it cannot be reading sequence.
* **9SHY is not a contradiction.** Its anchor-contacting nucleotide (auth 19) is absent from the
  model. The rule returns NOT RESOLVED so that missing data cannot be read as a negative result.
* **The discriminating experiment is still kinetic**, and it is preregistered in
  `results/solve/PREREGISTRATION_WP14.md` and gated by `results/solve/WP15_DESIGN_VALIDATION.md`.

The one experiment this result does raise: a **PLYQ cavity mutant (Y176A)**, tested for
processivity and for TPP1 stimulation. If the anchor is load-bearing, removing it should cost
processivity while leaving recruitment intact.
